# DBSCAN — Density-Based Spatial Clustering

## What is DBSCAN?
**DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** groups points that are closely packed together and marks points in low-density regions as **noise (outliers)**.

### Key Idea
> A cluster is a dense region of points separated from other dense regions by sparse regions.

### Two Hyperparameters
| Parameter | Meaning |
|-----------|---------|
| `eps` (ε) | Neighbourhood radius — how far to look for neighbours |
| `min_samples` | Minimum points within ε to be a core point |

### Three Types of Points
| Type | Condition | Role |
|------|-----------|------|
| **Core point** | ≥ `min_samples` points within ε | Forms the dense interior of a cluster |
| **Border point** | < `min_samples` within ε, but within ε of a core point | On the edge of a cluster |
| **Noise point** | Neither core nor border | Labelled **-1** (outlier) |

### DBSCAN vs K-Means
| | K-Means | DBSCAN |
|--|---------|--------|
| Need to specify K | Yes | No |
| Handles noise/outliers | No | Yes (label = -1) |
| Cluster shape | Spherical only | Arbitrary shape |
| Sensitive to density variation | — | Yes (struggles with varying density) |
| Speed | Fast O(n) | Slower O(n log n) with index |

### This Notebook
1. Tiny demo — 6 points, show label assignment
2. Concentric circles — where K-Means fails but DBSCAN works


## Part 1 — Tiny Demo (6 Points)

### Step 1: Imports & Data

In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np
import matplotlib.pyplot as plt

# 6 points: two tight groups + one far outlier
X = np.array([[1, 2], [2, 2], [2, 3],
              [8, 7], [8, 8],
              [25, 80]])  # outlier

print("Points:")
for i, p in enumerate(X):
    print(f"  {i}: {p}")

### Step 2: Visualise Raw Points

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], s=100, edgecolors='k')
for i, p in enumerate(X):
    plt.annotate(f'P{i} {tuple(p)}', p, textcoords="offset points", xytext=(5, 5))
plt.title("6 Points — Before DBSCAN")
plt.xlabel("X")
plt.ylabel("Y")
plt.tight_layout()
plt.show()

### Step 3: Fit DBSCAN
**Parameters chosen:**
- `eps=10` — a point is a neighbour if within distance 10
- `min_samples=2` — need at least 2 neighbours to be a core point

**Expected result:**
- Group [1,2],[2,2],[2,3] → cluster 0 (close together, within eps=10)
- Group [8,7],[8,8] → cluster 1 (close to each other, within eps of each other)
- Point [25,80] → **-1 (noise/outlier)** — too far from any other point


In [ ]:
db = DBSCAN(eps=10, min_samples=2)
db.fit(X)
print(f"Cluster labels: {db.labels_}")
print()
for i, (point, label) in enumerate(zip(X, db.labels_)):
    status = 'NOISE' if label == -1 else f'Cluster {label}'
    print(f"  P{i} {tuple(point)} → {status}")

### Step 4: Visualise DBSCAN Labels

In [ ]:
colors = {0: 'blue', 1: 'green', -1: 'red'}
label_names = {0: 'Cluster 0', 1: 'Cluster 1', -1: 'Noise (outlier)'}

plt.figure(figsize=(7, 5))
for label in np.unique(db.labels_):
    mask = db.labels_ == label
    plt.scatter(X[mask, 0], X[mask, 1],
                c=colors[label], s=150, edgecolors='k',
                label=label_names[label])
plt.title(f"DBSCAN (eps=10, min_samples=2) — 6 Points")
plt.legend()
plt.tight_layout()
plt.show()

---
## Part 2 — Concentric Circles (Where K-Means Fails)

### Why K-Means Fails on Non-Spherical Data
K-Means assumes clusters are **convex and roughly spherical** (it minimises distance to centroid).
For concentric circles, the two rings share the same centroid → K-Means splits them incorrectly.

**DBSCAN** works by density, not distance to centroid → correctly separates the two rings.


### Step 5: Generate Concentric Circles

In [ ]:
from sklearn.datasets import make_circles

X_circles, _ = make_circles(n_samples=500, factor=0.5, noise=0.03, random_state=4)

plt.figure(figsize=(6, 5))
plt.scatter(X_circles[:, 0], X_circles[:, 1], s=20, edgecolors='k', linewidths=0.3)
plt.title("Concentric Circles — Raw Data")
plt.tight_layout()
plt.show()

### Step 6: K-Means on Concentric Circles — Fails
K-Means splits the data vertically/horizontally — cannot detect ring structure.


In [ ]:
from sklearn.cluster import KMeans

km_circles = KMeans(n_clusters=2, random_state=42, n_init=10)
km_labels = km_circles.fit_predict(X_circles)

plt.figure(figsize=(6, 5))
plt.scatter(X_circles[:, 0], X_circles[:, 1], c=km_labels, cmap='bwr', s=20, edgecolors='k', linewidths=0.2)
plt.title("K-Means on Circles — WRONG (splits vertically)")
plt.tight_layout()
plt.show()

### Step 7: DBSCAN on Concentric Circles — Works
**Parameters:**
- `eps=0.1` — tight neighbourhood radius (circles are close but distinct)
- `min_samples=5` — need 5 neighbours to be a core point

DBSCAN follows the density of each ring and correctly assigns them to separate clusters.


In [ ]:
dbscan = DBSCAN(eps=0.1, min_samples=5)
clusters = dbscan.fit_predict(X_circles)

n_clusters = len(set(clusters)) - (1 if -1 in clusters else 0)
n_noise = list(clusters).count(-1)
print(f"Clusters found: {n_clusters}")
print(f"Noise points:   {n_noise}")

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X_circles[:, 0], X_circles[:, 1],
            c=clusters, cmap='viridis', s=20, edgecolors='k', linewidths=0.2)
plt.title(f"DBSCAN on Circles — CORRECT (eps=0.1, min_samples=5)")
plt.xlabel("Feature 0")
plt.ylabel("Feature 1")
plt.colorbar(label='Cluster (-1 = noise)')
plt.tight_layout()
plt.show()

## Step 8: Effect of eps — Hyperparameter Sensitivity
`eps` is the most critical parameter. Too small → everything becomes noise. Too large → everything merges.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
eps_values = [0.05, 0.1, 0.3]

for ax, eps in zip(axes, eps_values):
    db_tmp = DBSCAN(eps=eps, min_samples=5)
    lbl = db_tmp.fit_predict(X_circles)
    n_c = len(set(lbl)) - (1 if -1 in lbl else 0)
    n_n = list(lbl).count(-1)
    ax.scatter(X_circles[:, 0], X_circles[:, 1], c=lbl, cmap='tab10', s=10, edgecolors='none')
    ax.set_title(f"eps={eps}\n{n_c} clusters, {n_n} noise pts")
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("DBSCAN — Effect of eps (min_samples=5)", y=1.02)
plt.tight_layout()
plt.show()

## Summary

```
DBSCAN Key Points:

  Parameters:
    eps          → neighbourhood radius (tune with k-distance plot)
    min_samples  → min neighbours to be a core point (rule of thumb: 2×features)

  Labels:
    0, 1, 2, ... → cluster IDs
    -1           → noise / outlier

  Strengths:
    ✓ Finds arbitrary-shaped clusters
    ✓ Detects outliers automatically (label = -1)
    ✓ No need to specify K

  Weaknesses:
    ✗ Struggles with varying-density clusters
    ✗ High-dimensional data (eps hard to set)
    ✗ Slow on very large datasets without spatial index

  When to use:
    → Geospatial clustering (crime hotspots, GPS traces)
    → Anomaly/outlier detection
    → Non-spherical cluster shapes
    → When K is unknown and outliers exist

  sklearn:
    DBSCAN(eps=0.1, min_samples=5).fit_predict(X)
```
